# Laboratorio: Modelos del lenguaje con RNNs

En este laboratorio, vamos a entrenar un modelo del lenguaje basado en caracteres con Recurrent Neural Networks. Asimismo, utilizaremos el modelo para generar texto. En particular, alimentaremos nuestro modelo con obras de la literatura clásica en castellano para obtener una red neuronal que sea capaz de "escribir" fragmentos literarios.

Los entrenamientos en esta laboratorio para obtener un modelo de calidad podrían tomar cierto tiempo (5-10 minutos por epoch), por lo que se aconseja empezar a trabajar pronto. El uso de GPUs no ayuda tanto con LSTMs como con CNNs, por lo que si tenéis máquinas potentes en casa es posible que podáis entrenar más rápido o a la misma velocidad que en Colab. En todo caso, la potencia de Colab es más que suficiente para completar este laboratorio con éxito.

<center><img src="https://upload.wikimedia.org/wikipedia/commons/d/d8/El_ingenioso_hidalgo_don_Quijote_de_la_Mancha.jpg" style="text-align: center" height="300px"></center>

El dataset a utilizar consistirá en un archivo de texto con el contenido íntegro en castellano antiguo de El Ingenioso Hidalgo Don Quijote de la Mancha, disponible de manera libre en la página de [Project Gutenberg](https://www.gutenberg.org). Asimismo, como apartado optativo en este laboratorio se pueden utilizar otras fuentes de texto. Aquí podéis descargar los datos a utilizar de El Quijote y un par de obras adicionales:

[El ingenioso hidalgo Don Quijote de la Mancha (Miguel de Cervantes)](https://onedrive.live.com/download?cid=C506CF0A4F373B0F&resid=C506CF0A4F373B0F%219424&authkey=AH0gb-qSo5Xd7Io)

[Compilación de obras teatrales (Calderón de la Barca)](https://onedrive.live.com/download?cid=C506CF0A4F373B0F&resid=C506CF0A4F373B0F%219433&authkey=AKvGD6DC3IRBqmc)

[Trafalgar (Benito Pérez Galdós)](https://onedrive.live.com/download?cid=C506CF0A4F373B0F&resid=C506CF0A4F373B0F%219434&authkey=AErPCAtMKOI5tYQ)

Como ya deberíamos de estar acostumbrados en problemas de Machine Learning, es importante echar un vistazo a los datos antes de empezar.

## 1. Carga y procesado del texto

Primero, vamos a descargar el libro e inspeccionar los datos. El fichero a descargar es una versión en .txt del libro de Don Quijote, a la cual se le han borrado introducciones, licencias y otras secciones para dejarlo con el contenido real de la novela.

In [1]:
import numpy as np 
import keras
import matplotlib.pyplot as plt
from keras.callbacks import LambdaCallback
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
import random
import io

path = keras.utils.get_file(
    fname="don_quijote.txt", 
    origin="https://onedrive.live.com/download?cid=C506CF0A4F373B0F&resid=C506CF0A4F373B0F%219424&authkey=AH0gb-qSo5Xd7Io"
)

Using TensorFlow backend.


Una vez descargado, vamos a leer el contenido del fichero en una variable. Adicionalmente, convertiremos el contenido del texto a minúsculas para ponérselo un poco más fácil a nuestro modelo (de modo que todas las letras sean minúsculas y el modelo no necesite diferenciar entre minúsculas y mayúsculas).

**1.1.** Leer todo el contenido del fichero en una única variable ***text*** y convertir el string a minúsculas

In [2]:
## RESPUESTA
def leer_fichero(path_fichero):
    with open(path_fichero, 'r') as file:
        return file.read().lower()

text_quij = leer_fichero(path)

Podemos comprobar ahora que efectivamente nuestra variable contiene el resultado deseado, con el comienzo tan característico del Quijote.

In [3]:
print("Longitud del texto: {}".format(len(text_quij)))
print(text_quij[0:300])

Longitud del texto: 2071198
capítulo primero. que trata de la condición y ejercicio del famoso hidalgo
don quijote de la mancha


en un lugar de la mancha, de cuyo nombre no quiero acordarme, no ha mucho
tiempo que vivía un hidalgo de los de lanza en astillero, adarga antigua,
rocín flaco y galgo corredor. una olla de algo más


## 2. Procesado de los datos

Una de las grandes ventajas de trabajar con modelos que utilizan caracteres en vez de palabras es que no necesitamos tokenizar el texto (partirlo palabra a palabra). Nuestro modelo funcionará directamente con los caracteres en el texto, incluyendo espacios, saltos de línea, etc.

Antes de hacer nada, necesitamos procesar el texto en entradas y salidas compatibles con nuestro modelo. Como sabemos, un modelo del lenguaje con RNNs acepta una serie de caracteres y predice el siguiente carácter en la secuencia.

* "*El ingenioso don Qui*" -> predicción: **j**
* "*El ingenioso don Quij*" -> predicción: **o**

De modo que la entrada y la salida de nuestro modelo necesita ser algo parecido a este esquema. En este punto, podríamos usar dos formas de preparar los datos para nuestro modelo.

1. **Secuencia a secuencia**. La entrada de nuestro modelo sería una secuencia y la salida sería esa secuencia trasladada un caracter a la derecha, de modo que en cada instante de tiempo la RNN tiene que predecir el carácter siguiente. Por ejemplo:

>* *Input*:   El ingenioso don Quijot 
>* *Output*: l ingenioso don Quijote

2. **Secuencia a carácter**. En este variante, pasaríamos una secuencia de caracteres por nuestra RNN y, al llegar al final de la secuencia, predeciríamos el siguiente carácter.

>* *Input*:   El ingenioso don Quijot 
>* *Output*: e

En este laboratorio, por simplicidad, vamos a utilizar la segunda variante.

De este modo, a partir del texto, hemos de generar nuestro propio training data que consista en secuencias de caracteres con el siguiente carácter a predecir. Para estandarizar las cosas, utilizaremos secuencias de tamaño *SEQ_LENGTH* caracteres (un hiperparámetro que podemos elegir nosotros).



#### 2.1. Obtención de los caracteres y mapas de caracteres

Antes que nada, necesitamos saber qué caracteres aparecen en el texto, ya que tendremos que diferenciarlos mediante un índice de 0 a *num_chars* - 1 en el modelo. Obtener:
 

1.   Número de caracteres únicos que aparecen en el texto.
2.   Diccionario que asocia char a índice único entre 0 y *num_chars* - 1. Por ejemplo, {'a': 0, 'b': 1, ...}
3.   Diccionario reverso de índices a caracteres: {0: 'a', 1: 'b', ...}


In [4]:
## RESPUESTA
from dataclasses import dataclass, field, InitVar

@dataclass
class Traductor:
    """
    Clase para codificar y decodificar caracteres
    """
    _encoder: dict = field(default_factory=dict, init=False)
    _decoder: dict = field(default_factory=dict, init=False)
    _numero_elementos: int = field(init=False)

    text: InitVar[str] = None

    def __post_init__(self, text: str):
        # Obtención de caracteres únicos
        characters = set()
        for char in text:
            characters |= {char}
        characters = sorted(characters)
        self._numero_elementos = len(characters)

        # Diccionarios de índices
        for char, idx in zip(characters, range(0, self._numero_elementos)):
            self._encoder[char] = idx
            self._decoder[idx] = char
    
    def numero_caracteres(self) -> int:
        """
        Devuelve el número de elementos distintos
        """
        return self._numero_elementos
    
    def codificar(self, char: str):
        """
        Método para convertir un character o una cadena de caracteres
        en sus correspondientes códigos
        """
        encoded = []
        for _char in char:
            code = self._encoder[_char] if _char in self._encoder else None
            encoded.append(code)
        
        if len(encoded) == 1:
            return encoded.pop()
        
        return encoded

    def decodificar(self, code: int):
        """
        Método para convertir un código en su correspondiente caracter
        """
        if code not in self._decoder:
            return None
        
        return self._decoder[code]

traductor_quij = Traductor(text_quij)

#### 2.2. Obtención de secuencias de entrada y carácter a predecir

Ahora, vamos a obtener las secuencias de entrada en formato texto y los correspondientes caracteres a predecir. Para ello, recorrer el texto completo leído anteriormente, obteniendo una secuencia de SEQ_LENGTH caracteres y el siguiente caracter a predecir. Una vez hecho, desplazarse un carácter a la izquierda y hacer lo mismo para obtener una nueva secuencia y predicción. Guardar las secuencias en una variable ***sequences*** y los caracteres a predecir en una variable ***next_chars***.

Por ejemplo, si el texto fuera "Don Quijote" y SEQ_LENGTH fuese 5, tendríamos

* *sequences* = ["Don Q", "on Qu", "n Qui", " Quij", "Quijo", "uijot"]
* *next_chars* = ['u', 'i', 'j', 'o', 't', 'e']

In [5]:
# Definimos el tamaño de las secuencias. Puedes dejar este valor por defecto.
SEQ_LENGTH = 30

## RESPUESTA
def generar_secuencias(text: str, seq_length: int = SEQ_LENGTH):
    sequences = []
    next_chars = []
    
    # Se itera el texto completo para obtener las secuencias y los caracteres finales asociados a cada secuencia
    for inicio in range(0, len(text) - seq_length):
        fin = inicio + seq_length
        sequences.append(text[inicio:fin])
        next_chars.append(text[fin])
        
    return sequences, next_chars

sequences_quij, next_chars_quij = generar_secuencias(text_quij)

Indicar el tamaño del training set que acabamos de generar.

In [6]:
## RESPUESTA
print(f"Número de secuencias generadas: {len(sequences_quij)}")

Número de secuencias generadas: 2071168


Como el Quijote es muy largo y tenemos muchas secuencias, podríamos encontrar problemas de memoria. Por ello, vamos a elegir un número máximo de ellas. Si estás corriendo esto localmente y tienes problemas de memoria, puedes reducir el tamaño aún más, pero ten cuidado porque, a menos datos, peor calidad del modelo.

In [7]:
porc_sequences = 0.5
max_sequences_quij = int(len(sequences_quij) * porc_sequences)

def reducir_secuencias(sequences: [str], next_chars: [str], max_seq: int):
    perm = np.random.permutation(len(sequences))
    sequences, next_chars = np.array(sequences), np.array(next_chars)
    sequences, next_chars = sequences[perm], next_chars[perm]
    sequences, next_chars = list(sequences[:max_seq]), list(next_chars[:max_seq])
    
    return sequences, next_chars
    
sequences_quij, next_chars_quij = reducir_secuencias(sequences_quij, next_chars_quij, max_sequences_quij)

print(f"Número de secuencias conservadas: {len(sequences_quij)}")

Número de secuencias conservadas: 1035584


#### 2.3. Obtención de input X y output y para el modelo

Finalmente, a partir de los datos de entrenamiento que hemos generado vamos a crear los arrays de datos X e y que pasaremos a nuestro modelo.

Para ello, vamos a utilizar *one-hot encoding* para nuestros caracteres. Por ejemplo, si sólo tuviéramos 4 caracteres (a, b, c, d), las representaciones serían: (1, 0, 0, 0), (0, 1, 0, 0), (0, 0, 1, 0) y (0, 0, 0, 1).

De este modo, **X** tendrá shape *(num_sequences, seq_length, num_chars)* e **y** tendrá shape *(num_sequences, num_chars)*. 



In [8]:
def generar_conjunto_entrenamiento(sequences: [str],
                                   next_chars: [str],
                                   traductor: Traductor,
                                   seq_length: int = SEQ_LENGTH):
    num_sequences = len(sequences)
    num_chars = traductor.numero_caracteres()
    
    X = np.zeros((num_sequences, seq_length, num_chars))
    y = np.zeros((num_sequences, num_chars))

    ## Tu código para rellenar X e y aquí. Pista: utilizar el diccionario de
    ## chars a índices obtenido anteriormente junto con numpy. Por ejemplo,
    ## si hacemos 
    ##     X[0, 1, char_to_indices['a']] = 1
    ## estamos diciendo que para la segunda posición de la primera secuencia se
    ## tiene una 'a'

    ## RESPUESTA
    for sequence, next_char, idx_seq in zip(sequences, next_chars, range(0, num_sequences)):
        for char, idx_char in zip(sequence, range(0, seq_length)):
            X[idx_seq, idx_char, traductor.codificar(char)] = 1
        y[idx_seq, traductor.codificar(next_char)] = 1
        
    return X, y
        
X_quij, y_quij = generar_conjunto_entrenamiento(sequences_quij, next_chars_quij, traductor_quij)

## 3. Definición del modelo y entrenamiento

Una vez tenemos ya todo preparado, es hora de definir el modelo. Define un modelo que utilice una **LSTM** con **128 unidades internas**. Si bien el modelo puede definirse de una manera más compleja, para empezar debería bastar con una LSTM más una capa Dense con el *softmax* que predice el siguiente caracter a producir. Adam puede ser una buena elección de optimizador.

Una vez el modelo esté definido, entrénalo un poco para asegurarte de que la loss es decreciente. No es necesario guardar la salida de este entrenamiento en el entregable final, ya que vamos a hacer el entrenamiento más informativo en el siguiente punto.

In [9]:
## RESPUESTA
# Parámetros de entrenamiento
EPOCHS = 10
BATCH_SIZE = 128
VALIDATION_SPLIT = 0.2

OPTIMIZER = 'adam'
LOSS = 'categorical_crossentropy'
METRICS = ['accuracy']

num_chars_quij = traductor_quij.numero_caracteres()

# Creación y entrenamiento inicial del modelo
model_lstm = Sequential()

model_lstm.add(LSTM(units=128, dropout=0.2, recurrent_dropout=0.2))
model_lstm.add(Dense(num_chars_quij, activation='softmax'))

# Compilación del modelo
model_lstm.compile(optimizer=OPTIMIZER, loss=LOSS, metrics=METRICS)

# Entrenamiento del modelo
history = model_lstm.fit(X_quij, y_quij,
                         epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         validation_split=VALIDATION_SPLIT)

Train on 828467 samples, validate on 207117 samples
Epoch 1/10
828467/828467 [==============================] - 378s 456us/step - loss: 2.2920 - accuracy: 0.3132 - val_loss: 1.9626 - val_accuracy: 0.3918
Epoch 2/10
828467/828467 [==============================] - 391s 472us/step - loss: 2.0718 - accuracy: 0.3662 - val_loss: 1.8087 - val_accuracy: 0.4429
Epoch 3/10
828467/828467 [==============================] - 399s 481us/step - loss: 1.9908 - accuracy: 0.3904 - val_loss: 1.7220 - val_accuracy: 0.4654
Epoch 4/10
828467/828467 [==============================] - 393s 474us/step - loss: 1.9403 - accuracy: 0.4042 - val_loss: 1.6710 - val_accuracy: 0.4805
Epoch 5/10
828467/828467 [==============================] - 398s 480us/step - loss: 1.9060 - accuracy: 0.4140 - val_loss: 1.6303 - val_accuracy: 0.4920
Epoch 6/10
828467/828467 [==============================] - 389s 470us/step - loss: 1.8812 - accuracy: 0.4214 - val_loss: 1.6028 - val_accuracy: 0.5016
Epoch 7/10
828467/828467 [==========

Para ver cómo evoluciona nuestro modelo del lenguaje, vamos a generar texto según va entrenando. Para ello, vamos a programar una función que, utilizando el modelo en su estado actual, genere texto, con la idea de ver cómo se va generando texto al entrenar cada epoch.

En el código de abajo podemos ver una función auxiliar para obtener valores de una distribución multinomial. Esta función se usará para muestrear el siguiente carácter a utilizar según las probabilidades de la salida de softmax (en vez de tomar directamente el valor con la máxima probabilidad, obtenemos un valor aleatorio según la distribución de probabilidad dada por softmax, de modo que nuestros resultados serán más diversos, pero seguirán teniendo "sentido" ya que el modelo tenderá a seleccionar valores con más probabilidad).



In [10]:
def sample(probs, temperature=1.0):
    """Nos da el índice del elemento a elegir según la distribución
    de probabilidad dada por probs.
    
    Args:
      probs es la salida dada por una capa softmax:
        probs = model.predict(x_to_predict)[0]
      
      temperature es un parámetro que nos permite obtener mayor
        "diversidad" a la hora de obtener resultados. 
        
        temperature = 1 nos da la distribución normal de softmax
        0 < temperature < 1 hace que el sampling sea más conservador,
          de modo que sampleamos cosas de las que estamos más seguros
        temperature > 1 hace que los samplings sean más atrevidos,
          eligiendo en más ocasiones clases con baja probabilidad.
          Con esto, tenemos mayor diversidad pero se cometen más
          errores.
    """
    # Cast a float64 por motivos numéricos
    probs = np.asarray(probs).astype('float64')
    
    # Hacemos logaritmo de probabilidades y aplicamos reducción
    # por temperatura.
    probs = np.log(probs) / temperature
    
    # Volvemos a aplicar exponencial y normalizamos de nuevo
    exp_probs = np.exp(probs)
    probs = exp_probs / np.sum(exp_probs)
    
    # Hacemos el sampling dadas las nuevas probabilidades
    # de salida (ver doc. de np.random.multinomial)
    samples = np.random.multinomial(1, probs, 1)
    return np.argmax(samples)


Utilizando la función anterior y el modelo entrenado, vamos a añadir un callback a nuestro modelo para que, según vaya entrenando, veamos los valores que resultan de generar textos con distintas temperaturas al acabar cada epoch.

Para ello, abajo tenéis disponible el callback *on_epoch_end*. Esta función elige una secuencia de texto al azar en el texto disponible en la variable
text y genera textos de longitud *GENERATED_TEXT_LENGTH* según las temperaturas en *TEMPERATURES_TO_TRY*, utilizando para ello la función *generate_text*.

Completa la función *generate_text* de modo que utilicemos el modelo y la función sample para generar texto.

NOTA: Cuando hagas model.predict, es aconsejable usar verbose=0 como argumento para evitar que la función imprima valores de salida.

In [11]:
## RESPUESTA
# Callback para generar texto durante el entrenamiento
class TextGenerator(keras.callbacks.Callback):
    def __init__(self,
                 traductor: Traductor,
                 text: str,
                 seq_length: int = SEQ_LENGTH,
                 temperatures_to_try: [float] = [0.2, 0.5, 1.0, 1.2],
                 generated_text_length: int = 300,
                 **kwargs):
        
        super(TextGenerator, self).__init__()
        
        self.traductor = traductor
        self.text = text
        self.seq_length = seq_length
        self.temperatures_to_try = temperatures_to_try
        self.generated_text_length = generated_text_length

    def on_epoch_end(self, epoch, logs):
        print("\n\n\n")
        
        # Primero, seleccionamos una secuencia al azar para empezar a predecir
        # a partir de ella
        start_pos = random.randint(0, len(self.text) - self.seq_length - 1)
        seed_text = self.text[start_pos:start_pos + self.seq_length]
        for temperature in self.temperatures_to_try:
            print("------> Epoch: {} - Generando texto con temperature {}".format(
                  epoch + 1, temperature))
            
            generated_text = self._generate_text(seed_text, self.model, 
                                                 self.generated_text_length,
                                                 temperature)
            print("Seed: {}".format(seed_text))
            print("Texto generado: {}".format(generated_text))
            print()
            
    def _generate_text(self, seed_text, model, length, temperature=1):
        """Genera una secuencia de texto a partir de seed_text utilizando model.

        La secuencia tiene longitud length y el sampling se hace con la temperature
        definida.
        """

        # Aquí guardaremos nuestro texto generado, que incluirá el
        # texto origen
        generated = seed_text

        # Utilizar el modelo en un bucle de manera que generemos
        # carácter a carácter. Habrá que construir los valores de
        # X_pred de manera similar a como hemos hecho arriba, salvo que
        # aquí sólo se necesita una oración
        # Nótese que el x que utilicemos tiene que irse actualizando con
        # los caracteres que se van generando. La secuencia de entrada al
        # modelo tiene que ser una secuencia de tamaño SEQ_LENGTH que
        # incluya el último caracter predicho.

        ### INICIO RESPUESTA
        num_chars = self.traductor.numero_caracteres()
        while (len(generated) < length):
            x = np.zeros((1, self.seq_length, num_chars))

            sequence = generated[-self.seq_length:]
            for char, idx_char in zip(sequence, range(0, self.seq_length)):
                x[0, idx_char, self.traductor.codificar(char)] = 1

            y = model.predict(x, verbose=False)[0]
            nuevo_char = sample(y, temperature)
            generated += self.traductor.decodificar(nuevo_char)
        ### FIN RESPUESTA

        return generated

Entrena ahora tu modelo. No te olvides de añadir *generation_callback* a la lista de callbacks utilizados en fit(). Ya que las métricas de clasificación no son tan críticas aquí (no nos importa tanto acertar el carácter exacto, sino obtener una distribución de probabilidad adecuada), no es necesario monitorizar la accuracy ni usar validation data, si bien puedes añadirlos para asegurarte de que todo está en orden.


In [12]:
## RESPUESTA
# Entrenamiento del modelo con callback
history = model_lstm.fit(X_quij, y_quij,
                         epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         validation_split=VALIDATION_SPLIT,
                         callbacks=[ TextGenerator(traductor_quij, text_quij) ])

Train on 828467 samples, validate on 207117 samples
Epoch 1/10
828467/828467 [==============================] - 391s 471us/step - loss: 1.8142 - accuracy: 0.4403 - val_loss: 1.5209 - val_accuracy: 0.5250




------> Epoch: 1 - Generando texto con temperature 0.2
Seed:  húmida eco, que le respondies
Texto generado:  húmida eco, que le respondiese a la mano, que no había de la parte de la caballería y de la panza, que en el caballero que le parece de la mano, y que no se le dijo:

-¿qué de la mano -respondió don quijote-, que no había de la manos de la caballería que no se le había para de la manos de la caball

------> Epoch: 1 - Generando texto con temperature 0.5
Seed:  húmida eco, que le respondies
Texto generado:  húmida eco, que le respondiesen, y en el mano, con gusto en la mandad de la jumpa. ma dijo:

-¡vuestro merced -respondió sancho-, porque el caballero me la le respondió sancho, se no de la libra de la manohe presuntar de la duquesa de los cuales de muchas de caballero y qu

Seed:  estraño proceder, teniéndole 
Texto generado:  estraño proceder, teniéndole dinfe de los demoros
moscañadas, así, yo trevamos o persuas; porque, si podregre dar su duquesa le inprimera, si él tu pormen
de sus
digos de sue promedidos bescades,
que es la
leyuderle las para nanzas de si lerpa.

-¡ahí y, ¡distora se or naujamos de si
laladura.

-¡a

------> Epoch: 5 - Generando texto con temperature 1.2
Seed:  estraño proceder, teniéndole 
Texto generado:  estraño proceder, teniéndole cudísio traigo, funo
recortes te sues queer discrecho de dao los ariatos, dámo en ensolar decto a mon estas vechos,
mey verías l squís for aresinas partenos pendís
deles a morcienes que donde si y mozparto herra cominera que queraa pesadarica y mano inumpece; por corntó

Epoch 6/10
828467/828467 [==============================] - 391s 472us/step - loss: 1.7802 - accuracy: 0.4493 - val_loss: 1.4781 - val_accuracy: 0.5360




------> Epoch: 6 - Generando texto con temperature 0.2
Seed: o quieras hacer má

Seed:  y las dueñas, madre y hija, d
Texto generado:  y las dueñas, madre y hija, de caballeros que yo se desperaron don quijote de la hacienes de la aguada, que la he visto en ellos de la caballería, y en la
casa. y así que había por estaban en la verdad y de don quijote y mi amigo, se fuera de la mancha puesta a mi padre, porque está las males de aq

------> Epoch: 10 - Generando texto con temperature 1.0
Seed:  y las dueñas, madre y hija, d
Texto generado:  y las dueñas, madre y hija, dejando de mis estarros, de la gderistza perruan, y tan mía, y llanos, y tales de la partesa de haces,
y modos cantaria sema, porque la
compaño, y entrado en feamo, y dijo:

-¡a-sí así a no buena pale agora-, cuanto andas que
frese cada con eltor en
vivos vez basador; en

------> Epoch: 10 - Generando texto con temperature 1.2
Seed:  y las dueñas, madre y hija, d
Texto generado:  y las dueñas, madre y hija, desta, cornagunos. mañana, ejorto, la casa
    ó desno. fuera patece'; y no fuera ridelella, así

## Entregable

Completa los apartados anteriores para entrenar modelos del lenguaje que sean capaces de generar texto con cierto sentido. Comentar los resultados obtenidos y cómo el modelo va mejorando época a época. Comentar las diferencias apreciadas al utilizar diferentes valores de temperatura. Entregar al menos la salida de un entrenamiento completo con los textos generados época a época.

El objetivo no es conseguir generar pasajes literarios con coherencia, sino obtener lenguaje que se asemeje en cierta manera a lo visto en el texto original y donde las palabras sean reconocibles como construcciones en castellano. Como ejemplo de lo que se puede conseguir, este es el resultado de generar texto después de 10 epochs y con temperature 0.2:


```
-----> Epoch: 10 - Generando texto con temperature 0.2
Seed: o le cautivaron y rindieron el
Texto generado: o le cautivaron y rindieron el caballero de la caballería de la mano de la caballería del cual se le dijo:

-¿quién es el verdad de la caballería de la caballería de la caballería de la caballería de la caballería, y me ha de habían de la mano que el caballero de la mano de la caballería. y que no se le habían de la mano de la c

```

Asimismo, se proponen los siguientes aspectos opcionales para conseguir nota extra:

*   Experimentar con los textos de teatro en verso de Calderón de la Barca (¿es capaz el modelo de aprender las estructuras del teatro en verso?) o con alguno de los otros textos disponibles. También se puede probar con textos de vuestra elección.
*   Experimentar con distintos valores de SEQ_LENGTH.
*   Experimentar con los hiperparámetros del modelo o probar otro tipo de modelos como GRUs o *stacked* RNNs (RNNs apiladas).
*   Experimentar utilizando embeddings en vez de representaciones one-hot.
*   (Difícil) Entrenar un modelo secuencia a secuencia en vez de secuencia a carácter.




### Conclusiones

Como puede observarse, el generador de texto es capaz de generar palabras que en unos casos están mejor formadas que en otros, pero se ve cómo a medida que evolucionan las épocas de entrenamiento, los textos generados van mejorando y produciendo palabras más correctas, aunque desde luego el sentido de la frase dista mucho de tener sentido. Esto mejoraría seguramente en un modelo de secuencia a secuencia en lugar de secuencia a carácter.

### Prueba con GRUs

Ahora se realiza el mismo estudio entrenando el modelo con una arquitectura `GRU`

In [13]:
from keras.layers import GRU

# Descarga del texto
path = keras.utils.get_file(
    fname="calderon_de_la_barca.txt", 
    origin="https://onedrive.live.com/download?cid=C506CF0A4F373B0F&resid=C506CF0A4F373B0F%219433&authkey=AKvGD6DC3IRBqmc"
)

text_cal = leer_fichero(path)

print("Longitud del texto: {}".format(len(text_cal)))
print(f"\n\n{text_cal[100:400]}\n\n\n")

# Traductor del texto
traductor_cal = Traductor(text_cal)

# Secuencias del texto (no se reduce el número de secuencias porque es asumible)
sequences_cal, next_chars_cal = generar_secuencias(text_cal)

# Conjuntos de entrenamiento
X_cal, y_cal = generar_conjunto_entrenamiento(sequences_cal, next_chars_cal, traductor_cal)

# Creación y entrenamiento inicial del modelo
num_chars_cal = traductor_cal.numero_caracteres()
model_gru = Sequential()

model_gru.add(GRU(units=128, dropout=0.2, recurrent_dropout=0.2))
model_gru.add(Dense(num_chars_cal, activation='softmax'))

# Compilación del modelo
model_gru.compile(optimizer=OPTIMIZER, loss=LOSS, metrics=METRICS)

# Entrenamiento del modelo (sin generador)
history = model_gru.fit(X_cal, y_cal,
                        epochs=EPOCHS,
                        batch_size=BATCH_SIZE,
                        validation_split=VALIDATION_SPLIT)

# Continuación del entrenamiento del modelo (con generador)
history = model_gru.fit(X_cal, y_cal,
                        epochs=EPOCHS,
                        batch_size=BATCH_SIZE,
                        validation_split=VALIDATION_SPLIT,
                        callbacks=[ TextGenerator(traductor_cal, text_cal) ])

Longitud del texto: 401338


  segismundo, _príncipe_.
  astolfo, _duque de moscovia_.
  clotaldo, _viejo_.
  clarin, _gracioso_.
  estrella, _infanta_.
  rosaura, _dama_.
  _soldados._
  _guardas._
  _músicos._
  _acompañamiento._
  _criados._
  _damas._


la escena es en la corte de polonia, en una fortaleza poco distante y
e



Train on 321046 samples, validate on 80262 samples
Epoch 1/10
321046/321046 [==============================] - 135s 422us/step - loss: 2.3033 - accuracy: 0.3310 - val_loss: 2.0280 - val_accuracy: 0.3920
Epoch 2/10
321046/321046 [==============================] - 135s 422us/step - loss: 2.0180 - accuracy: 0.3879 - val_loss: 1.9178 - val_accuracy: 0.4189
Epoch 3/10
321046/321046 [==============================] - 134s 418us/step - loss: 1.9333 - accuracy: 0.4101 - val_loss: 1.8567 - val_accuracy: 0.4356
Epoch 4/10
321046/321046 [==============================] - 135s 421us/step - loss: 1.8801 - accuracy: 0.4240 - val_loss: 1.8181 - val_accuracy: 0.4443
Epoch 5/

Seed: 

                       sin m
Texto generado: 

                       sin mí! paca hobro que viva adminfr
  fansonde no vido,
  tú, en todo vedente viene y estana,
  te dacia ino
  que ignor y nacreve
  prente
  (¡vencosido aquel ya á tu escura haste curirsero
  de tí, tora! que dustir
  leciarte su podre denore, iptosa_.

cuardo,
  _mal, solt

------> Epoch: 4 - Generando texto con temperature 1.2
Seed: 

                       sin m
Texto generado: 

                       sin munez  de hanida
  ya her friz.

felio.


jemoe, quitintes;
  apor que tampe tu esaurqun ¡grancerdes landas,
  del hay mecio.
  vicion?


jolpendas.

eustina
  a ura ruiste; famas..

advinto mazoridos huya:
  ó duga vida. despedi lusgo a, el rubrtra. _(dija, cieralo.) cl

Epoch 5/10
321046/321046 [==============================] - 135s 420us/step - loss: 1.7130 - accuracy: 0.4706 - val_loss: 1.6514 - val_accuracy: 0.4938




------> Epoch: 5 - Generando texto con temperature 0.2
Seed: s dispongais mis c

Seed:  boca de la herida,
  escucha,
Texto generado:  boca de la herida,
  escucha, y esto?

curcio.

                                                                                                                                                                                                                                                           

------> Epoch: 9 - Generando texto con temperature 1.0
Seed:  boca de la herida,
  escucha,
Texto generado:  boca de la herida,
  escucha, si defentes ifácia sol distido?

gil.

                         mi cuagre? y _(hezudo.
  si ruyos.

rlcoricibio:
  rocir
  mure recirle ya preve.

pustina.

  un soere
  que versigillera,
  esa á la vengo, elorio_...

clarin.

leldamino examora sobrímas:
  clotado á es

------> Epoch: 9 - Generando texto con temperature 1.2
Seed:  boca de la herida,
  escucha,
Texto generado:  boca de la herida,
  escucha, centro, yos vertanor lelao,
  entenjer nabia, ¿quiante, cestá. ¿qué?

gil.

  demos consusigras?

### Conclusiones

Como puede verse, el texto generado intenta manetener la estructura del texto general, aunque de nuevo las palabras no dejan de ser inventadas en la mayoría de los casos. Con más entrenamiento y una arquitectura más sofisticada podría mejorarse el restulado, pero desde luego, la mejora notable se conseguiría con entrenamientos de secuencia a secuencia en lugar de secuencia a carácter.